In [27]:
import gradio as gr
from pydantic import BaseModel, Field, validate_call, ValidationError
from openai import OpenAI
from typing import Literal, Annotated

In [28]:
MODELS=['llama3.1:8b', 'gemma4:e4b', 'ministral-3:8b']

In [29]:
@validate_call
def calulcate_discount(
    price: Annotated[float, Field(gt=0)],
    discount_percent: Annotated[float, Field(ge=0, le=100)] = 10.0,
) -> float:
    return round(price * (1 - discount_percent / 100), 2)

In [32]:
try:
    result = calulcate_discount(30, 80)
    print(result)
except ValidationError as e:
    print(e)

6.0


In [33]:
class OrderInput(BaseModel):
    item_name: str = Field(min_length=1)
    quantity: int = Field(gt=0)
    unit_price: float = Field(gt=0)

class OrderSummary(BaseModel):
    item_name: str
    total: float
    is_bulk: bool

In [34]:
def process_order(order: OrderInput) -> OrderSummary:
    total = round(order.quantity * order.unit_price, 2)
    return OrderSummary(
        item_name=order.item_name,
        total=total,
        is_bulk=order.quantity >= 10,
    )

In [36]:
# Validate raw data into OrderInput, then pass to the function
incoming = OrderInput.model_validate({"item_name": "Notebook", "quantity": "15", "unit_price": "450"})
summary = process_order(incoming)
print(summary.model_dump_json(indent=2))

{
  "item_name": "Notebook",
  "total": 6750.0,
  "is_bulk": true
}
